# Motor-Pore Offset Analysis: Characterizing Amino Acid Signatures

**Issue**: [#81 - Implement motor-pore offset correction for amino acid signatures](https://github.com/rnabioco/leech/issues/81)

## Background

Nanopore sequencers have two detection sites:
1. **Motor (helicase)**: Controls translocation speed → affects **dwell time**
2. **Pore (reader head)**: Measures ionic current → affects **electrical signal**

These sites are separated by **12-13 nucleotides** physically. When an aminoacylated tRNA passes through:
- Amino acid in **motor** → increases dwell time for nucleotides currently in **pore** (~12-13 nt away)
- Amino acid in **pore** → distorts electrical current

**Current Problem**: The codebase incorrectly assumes dwell and current signatures are aligned to the same genomic position. This explains why **dwell features don't improve model accuracy** - they're extracted from the wrong position.

## Analysis Goals

1. **Extract wide-window features** (±30 nt from motif) for charged vs uncharged tRNAs
2. **Visualize position-wise signatures** to identify where dwell and current changes occur
3. **Measure offset empirically** - determine direction and magnitude
4. **Propose implementation strategy** for offset-corrected feature extraction

## Molecule Structure

```
5' ─── [tRNA ~73 nt] ─── CCA-[aa]-GGC ─── [adaptor ~37 nt, AAAA] ─── 3'
                             ↑
                       Motif: CCATGGC
                       (T = amino acid placeholder in reference)

Sequencing direction: 3' → 5' (adaptor enters pore first)
Reporting: 5' → 3' (standard convention)
```

In [ ]:
# Imports
import warnings
from pathlib import Path

import numpy as np
import plotnine as p9
import polars as pl
from scipy import signal as sp_signal
from scipy import stats

# leech imports
from leech.chunking.extractor import LeechRead, extract_training_chunks
from leech.features import (
    compute_dwell_times,
    compute_signal_features,
    extract_move_table,
    normalize_signal,
)
from leech.io.bam_reader import BAMReader
from leech.io.motif_search import get_motif_searcher
from leech.io.pod5_reader import POD5Reader
from leech.io.reference import get_reference_sequences

warnings.filterwarnings("ignore")

# Plotting aesthetics
p9.theme_set(p9.theme_minimal() + p9.theme(figure_size=(14, 6)))

print("✓ Imports successful")

## Configuration

Set paths to your POD5 and BAM files, and define extraction parameters.

In [ ]:
# ============================================================================
# USER CONFIGURATION - Update for your analysis
# ============================================================================

# Choose which samples to analyze
# Charged options: ala_synthetic, arg_synthetic, asn_synthetic, asp_synthetic,
#                  cys_synthetic_rep1, cys_synthetic_rep2, gln_synthetic,
#                  glu_synthetic, gly_synthetic, his_synthetic, ile_synthetic,
#                  leu_synthetic, lys_synthetic, met_synthetic, phe_synthetic,
#                  pro_synthetic, ser_synthetic, thr_synthetic, trp_synthetic,
#                  tyr_synthetic, val_synthetic
# Uncharged: uncharged_synthetic

CHARGED_SAMPLE = "ala_synthetic"  # Pick any charged amino acid sample
UNCHARGED_SAMPLE = "uncharged_synthetic"

# Pipeline paths (from pipeline/config/samples-alpine.yaml and config.yaml)
PROJECT_NAME = "synthetic-trna"  # From samples-alpine.yaml
BASE_DIR = Path("/scratch/alpine/jhesselberth@xsede.org/leech")

# Merged POD5 files (generated by pipeline: pod5/{sample}/{sample}.pod5)
CHARGED_POD5 = BASE_DIR / PROJECT_NAME / "pod5" / CHARGED_SAMPLE / f"{CHARGED_SAMPLE}.pod5"
UNCHARGED_POD5 = BASE_DIR / PROJECT_NAME / "pod5" / UNCHARGED_SAMPLE / f"{UNCHARGED_SAMPLE}.pod5"

# BAM files (generated by pipeline: bam/rebasecall/{sample}/{sample}.aligned.bam)
# Contains alignments with move tables (mv tag) for signal-to-sequence mapping
CHARGED_BAM = BASE_DIR / PROJECT_NAME / "bam" / "rebasecall" / CHARGED_SAMPLE / f"{CHARGED_SAMPLE}.aligned.bam"
UNCHARGED_BAM = BASE_DIR / PROJECT_NAME / "bam" / "rebasecall" / UNCHARGED_SAMPLE / f"{UNCHARGED_SAMPLE}.aligned.bam"

# Reference FASTA (for motif position identification)
# Used to find CCATGGC motif in reference coordinates
REFERENCE_FASTA = Path("../pipeline/resources/references/synthetic-trna.fa")

# Motif search parameters (from pipeline config)
MOTIF = "CCATGGC"  # CCA-[aa as T]-GGC (T is placeholder for amino acid)
MOTIF_OFFSET = 3  # Focus on position 3 (the 'T' = amino acid position)
MOTIF_REFERENCE = "fasta"  # Search in reference sequence (avoids basecalling bias)

# Wide-window extraction parameters
KMER_CONTEXT = 30  # Extract ±30 bases from motif (total 61 bases)
SIGNAL_CONTEXT = (600, 600)  # Signal context (samples left, right)

# Sampling parameters
MAX_READS_PER_CLASS = 500  # Number of reads to sample from each class
MIN_MAPQ = 10  # Minimum mapping quality (from pipeline)
RANDOM_SEED = 42  # For reproducibility

# Output directory
OUTPUT_DIR = Path("../output/offset_analysis") / f"{CHARGED_SAMPLE}_vs_{UNCHARGED_SAMPLE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("MOTOR-PORE OFFSET ANALYSIS CONFIGURATION")
print("=" * 80)
print(f"\nSamples:")
print(f"  Charged:    {CHARGED_SAMPLE}")
print(f"  Uncharged:  {UNCHARGED_SAMPLE}")
print(f"\nData sources (pipeline-generated files):")
print(f"  Charged POD5:   {CHARGED_POD5}")
print(f"  Charged BAM:    {CHARGED_BAM}")
print(f"  Uncharged POD5: {UNCHARGED_POD5}")
print(f"  Uncharged BAM:  {UNCHARGED_BAM}")
print(f"  Reference:      {REFERENCE_FASTA}")
print(f"\nData flow:")
print(f"  1. POD5  → Raw signal extraction")
print(f"  2. BAM   → Alignment + move tables (signal-to-sequence mapping)")
print(f"  3. FASTA → Motif position identification (CCATGGC)")
print(f"\nAnalysis parameters:")
print(f"  Motif:           {MOTIF} (offset={MOTIF_OFFSET}, reference-based)")
print(f"  K-mer context:   ±{KMER_CONTEXT} bases (total {2*KMER_CONTEXT+1} bases)")
print(f"  Signal context:  {SIGNAL_CONTEXT[0]}/{SIGNAL_CONTEXT[1]} samples")
print(f"  Max reads:       {MAX_READS_PER_CLASS} per class")
print(f"\nOutput:")
print(f"  Directory:       {OUTPUT_DIR}")
print("=" * 80)

# Verify files exist
print("\nVerifying data files...")
all_exist = True
for label, path in [
    ("Charged POD5", CHARGED_POD5),
    ("Charged BAM", CHARGED_BAM),
    ("Uncharged POD5", UNCHARGED_POD5),
    ("Uncharged BAM", UNCHARGED_BAM),
    ("Reference FASTA", REFERENCE_FASTA),
]:
    if path.exists():
        print(f"  ✓ {label}: {path}")
    else:
        print(f"  ✗ {label}: {path} (NOT FOUND)")
        all_exist = False

if not all_exist:
    print("\n⚠️  WARNING: Some files are missing!")
    print("   Make sure you've run the pipeline to generate POD5s and BAMs:")
    print("   1. Merge POD5s:     snakemake --profile profiles/slurm all_merge_pods")
    print("   2. Rebasecall:      snakemake --profile profiles/slurm all_rebasecall")
    print("   3. Align:           snakemake --profile profiles/slurm all_align")
else:
    print("\n✓ All data files found! Ready to proceed.")

## Step 1: Extract Wide-Window Features

Extract features across a wide window (±30 nt from motif) for both charged and uncharged reads.

In [ ]:
def extract_wide_window_features(pod5_path, bam_path, label_name, max_reads=500, reference_fasta=None):
    """
    Extract wide-window features for offset analysis.
    
    Returns:
        List of dicts with features at each position relative to motif.
    """
    print(f"\nExtracting features from {label_name}...")
    print(f"  POD5: {pod5_path}")
    print(f"  BAM:  {bam_path}")
    print(f"  Reference: {reference_fasta}")
    
    # Load reference sequences and create motif searcher
    reference_sequences = get_reference_sequences(bam_path, reference_fasta)
    print(f"  Loaded {len(reference_sequences)} reference sequences")
    
    motif_searcher = get_motif_searcher(
        mode=MOTIF_REFERENCE,
        reference_sequences=reference_sequences,
        skip_indels=True
    )
    
    chunks = []
    bam_reader = BAMReader(bam_path, min_mapq=MIN_MAPQ)
    pod5_reader = POD5Reader(pod5_path)
    
    n_processed = 0
    n_with_motif = 0
    n_errors = 0
    
    with bam_reader, pod5_reader:
        for aln in bam_reader.iter_alignments():
            if len(chunks) >= max_reads:
                break
                
            n_processed += 1
            
            if aln.is_unmapped or not aln.has_tag("mv") or aln.query_name is None:
                continue
            
            if aln.query_sequence is None:
                continue
                
            try:
                # Find motif positions using motif searcher
                motif_positions = motif_searcher.find_motif_positions(
                    read_id=aln.query_name,
                    sequence=aln.query_sequence,
                    alignment=aln,
                    motif=MOTIF
                )
                
                if not motif_positions:
                    continue
                    
                n_with_motif += 1
                
                # Get signal and features
                signal, _ = pod5_reader.get_signal(aln.query_name)
                signal_norm = normalize_signal(signal, method="median_mad")
                
                move_table = extract_move_table(aln)
                seq_to_sig_map = move_table.to_seq_to_sig_map()
                
                # Compute dwell times and signal features
                dwells = compute_dwell_times(move_table)
                # compute_signal_features returns dict with 'level_mean', 'level_median', 'level_std', 'level_range'
                signal_features = compute_signal_features(signal_norm, seq_to_sig_map)
                
                # Extract wide window around each motif occurrence
                for motif_pos in motif_positions:
                    focus_pos = motif_pos + MOTIF_OFFSET
                    
                    # Check boundaries
                    start_pos = focus_pos - KMER_CONTEXT
                    end_pos = focus_pos + KMER_CONTEXT + 1
                    
                    if start_pos < 0 or end_pos > len(dwells):
                        continue
                        
                    # Extract features across window
                    chunk = {
                        "read_id": aln.query_name,
                        "label": label_name,
                        "motif_pos": motif_pos,
                        "focus_pos": focus_pos,
                        "sequence": aln.query_sequence[start_pos:end_pos],
                        "dwells": dwells[start_pos:end_pos],
                        "signal_mean": signal_features["level_mean"][start_pos:end_pos],
                        "signal_std": signal_features["level_std"][start_pos:end_pos],
                        "signal_median": signal_features["level_median"][start_pos:end_pos],
                    }
                    chunks.append(chunk)
                    
                    if len(chunks) >= max_reads:
                        break
                        
            except Exception as e:
                n_errors += 1
                if n_errors <= 5:  # Only print first 5 errors
                    print(f"  Warning: Failed to process {aln.query_name}: {e}")
                continue
    
    if n_errors > 5:
        print(f"  ... and {n_errors - 5} more errors (suppressed)")
    
    print(f"  Processed {n_processed} alignments")
    print(f"  Found motif in {n_with_motif} reads")
    print(f"  Extracted {len(chunks)} wide-window chunks")
    
    return chunks


# Extract features for both classes
print("=" * 80)
print("EXTRACTING WIDE-WINDOW FEATURES")
print("=" * 80)

charged_chunks = extract_wide_window_features(
    CHARGED_POD5, CHARGED_BAM, "charged", 
    max_reads=MAX_READS_PER_CLASS,
    reference_fasta=REFERENCE_FASTA
)

uncharged_chunks = extract_wide_window_features(
    UNCHARGED_POD5, UNCHARGED_BAM, "uncharged",
    max_reads=MAX_READS_PER_CLASS,
    reference_fasta=REFERENCE_FASTA
)

print(f"\n✓ Total chunks extracted: {len(charged_chunks) + len(uncharged_chunks)}")

## Step 2: Compute Position-Wise Statistics

Aggregate features across all reads to compute mean and std at each position relative to the motif.

In [ ]:
def aggregate_position_wise_features(chunks, label):
    """
    Aggregate features across all chunks to get position-wise statistics.
    
    Returns:
        DataFrame with columns: position, feature, mean, std, sem, label
    """
    print(f"\nAggregating features for {label} ({len(chunks)} chunks)...")
    
    # Stack all features into arrays
    window_size = 2 * KMER_CONTEXT + 1
    
    dwells_matrix = np.array([c["dwells"] for c in chunks if len(c["dwells"]) == window_size])
    signal_mean_matrix = np.array([c["signal_mean"] for c in chunks if len(c["signal_mean"]) == window_size])
    signal_std_matrix = np.array([c["signal_std"] for c in chunks if len(c["signal_std"]) == window_size])
    signal_median_matrix = np.array([c["signal_median"] for c in chunks if len(c["signal_median"]) == window_size])
    
    print(f"  Valid chunks after filtering: {len(dwells_matrix)}")
    
    # Compute position-wise statistics
    positions = np.arange(-KMER_CONTEXT, KMER_CONTEXT + 1)  # Relative to motif focus
    
    data = []
    
    for pos_idx, pos in enumerate(positions):
        # Dwell
        data.append({
            "position": pos,
            "feature": "dwell_time",
            "mean": np.mean(dwells_matrix[:, pos_idx]),
            "std": np.std(dwells_matrix[:, pos_idx]),
            "sem": stats.sem(dwells_matrix[:, pos_idx]),
            "label": label,
        })
        
        # Signal mean
        data.append({
            "position": pos,
            "feature": "signal_mean",
            "mean": np.mean(signal_mean_matrix[:, pos_idx]),
            "std": np.std(signal_mean_matrix[:, pos_idx]),
            "sem": stats.sem(signal_mean_matrix[:, pos_idx]),
            "label": label,
        })
        
        # Signal std (variability)
        data.append({
            "position": pos,
            "feature": "signal_variability",
            "mean": np.mean(signal_std_matrix[:, pos_idx]),
            "std": np.std(signal_std_matrix[:, pos_idx]),
            "sem": stats.sem(signal_std_matrix[:, pos_idx]),
            "label": label,
        })
    
    return pl.DataFrame(data)


# Aggregate for both classes
print("=" * 80)
print("COMPUTING POSITION-WISE STATISTICS")
print("=" * 80)

charged_stats = aggregate_position_wise_features(charged_chunks, "charged")
uncharged_stats = aggregate_position_wise_features(uncharged_chunks, "uncharged")

# Combine
all_stats = pl.concat([charged_stats, uncharged_stats])

print(f"\n✓ Aggregated statistics: {len(all_stats)} rows")
print(f"  Features: {all_stats['feature'].unique().to_list()}")
print(f"  Position range: [{all_stats['position'].min()}, {all_stats['position'].max()}]")

# Preview
print("\nPreview (first 10 rows):")
print(all_stats.head(10))

## Step 3: Visualization - Position-Wise Feature Profiles

Plot dwell time and signal mean across position for charged vs uncharged.
This will reveal where the signatures appear relative to the motif (position 0).

In [ ]:
# Convert to pandas for plotting
stats_pd = all_stats.to_pandas()

# Create plots for each feature
features_to_plot = ["dwell_time", "signal_mean", "signal_variability"]
titles = {
    "dwell_time": "Dwell Time Across Position (Motor Signature)",
    "signal_mean": "Signal Mean Across Position (Pore Signature)",
    "signal_variability": "Signal Variability Across Position"
}

plots = []
for feature in features_to_plot:
    feature_data = stats_pd[stats_pd["feature"] == feature].copy()
    
    plot = (
        p9.ggplot(feature_data, p9.aes(x="position", y="mean", color="label", fill="label"))
        + p9.geom_line(size=1.2, alpha=0.8)
        + p9.geom_ribbon(p9.aes(ymin="mean - sem", ymax="mean + sem"), alpha=0.2, color=None)
        + p9.geom_vline(xintercept=0, linetype="dashed", color="black", alpha=0.5, size=1)
        + p9.geom_vline(xintercept=-13, linetype="dotted", color="gray", alpha=0.5)
        + p9.geom_vline(xintercept=13, linetype="dotted", color="gray", alpha=0.5)
        + p9.scale_color_manual(values={"charged": "#E74C3C", "uncharged": "#3498DB"})
        + p9.scale_fill_manual(values={"charged": "#E74C3C", "uncharged": "#3498DB"})
        + p9.labs(
            title=titles[feature],
            x="Position Relative to Motif (nt)",
            y=feature.replace("_", " ").title(),
            color="Sample",
            fill="Sample"
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(14, 5),
            plot_title=p9.element_text(size=12, weight="bold"),
            legend_position="right"
        )
    )
    plots.append(plot)
    
    # Save individual plot
    plot.save(OUTPUT_DIR / f"{feature}_profile.png", dpi=150, width=14, height=5)
    print(plot)

print(f"\n✓ Saved plots: {OUTPUT_DIR}/{{feature}}_profile.png")

## Step 4: Difference Plots - Charged - Uncharged

Plot the difference (charged - uncharged) to highlight where signatures are strongest.

In [ ]:
# Compute differences (charged - uncharged)
diff_data = []

for feature in features_to_plot:
    feature_data = stats_pd[stats_pd["feature"] == feature]
    
    charged_data = feature_data[feature_data["label"] == "charged"].sort_values("position")
    uncharged_data = feature_data[feature_data["label"] == "uncharged"].sort_values("position")
    
    positions = charged_data["position"].values
    diff = charged_data["mean"].values - uncharged_data["mean"].values
    
    for pos, d in zip(positions, diff):
        diff_data.append({
            "position": pos,
            "feature": feature,
            "difference": d
        })

diff_df = pl.DataFrame(diff_data).to_pandas()

# Create difference plots
feature_colors = {
    "dwell_time": "#E67E22",
    "signal_mean": "#8E44AD",
    "signal_variability": "#16A085"
}

diff_plots = []
for feature in features_to_plot:
    feature_diff = diff_df[diff_df["feature"] == feature].copy()
    
    # Add fill direction column
    feature_diff["fill_group"] = feature_diff["difference"].apply(
        lambda x: "Charged > Uncharged" if x >= 0 else "Uncharged > Charged"
    )
    
    plot = (
        p9.ggplot(feature_diff, p9.aes(x="position", y="difference"))
        + p9.geom_area(p9.aes(fill="fill_group"), alpha=0.3)
        + p9.geom_line(color=feature_colors[feature], size=1.2, alpha=0.8)
        + p9.geom_hline(yintercept=0, linetype="solid", color="black", alpha=0.5, size=0.8)
        + p9.geom_vline(xintercept=0, linetype="dashed", color="black", alpha=0.5, size=1)
        + p9.geom_vline(xintercept=-13, linetype="dotted", color="gray", alpha=0.5)
        + p9.geom_vline(xintercept=13, linetype="dotted", color="gray", alpha=0.5)
        + p9.scale_fill_manual(
            values={
                "Charged > Uncharged": feature_colors[feature],
                "Uncharged > Charged": "gray"
            }
        )
        + p9.labs(
            title=f"{titles[feature]} - Difference",
            x="Position Relative to Motif (nt)",
            y=f"Δ {feature.replace('_', ' ').title()}\n(Charged - Uncharged)",
            fill="Direction"
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(14, 5),
            plot_title=p9.element_text(size=12, weight="bold"),
            legend_position="right"
        )
    )
    diff_plots.append(plot)
    
    # Save plot
    plot.save(OUTPUT_DIR / f"{feature}_difference.png", dpi=150, width=14, height=5)
    print(plot)

print(f"\n✓ Saved difference plots: {OUTPUT_DIR}/{{feature}}_difference.png")

## Step 5: Peak Detection - Measure Offset

Automatically detect where dwell and current signatures peak, then calculate the offset.

In [ ]:
def find_signature_peak(diff_df, feature, window=(-20, 20)):
    """
    Find the position of maximum absolute difference for a feature.
    
    Args:
        diff_df: DataFrame with position and difference columns
        feature: Feature name to analyze
        window: (min_pos, max_pos) to search within
        
    Returns:
        dict with peak_position, peak_value, peak_type
    """
    feature_diff = diff_df[
        (diff_df["feature"] == feature) &
        (diff_df["position"] >= window[0]) &
        (diff_df["position"] <= window[1])
    ].copy()
    
    # Find position of maximum absolute difference
    feature_diff["abs_diff"] = feature_diff["difference"].abs()
    peak_idx = feature_diff["abs_diff"].idxmax()
    
    peak_row = feature_diff.loc[peak_idx]
    
    return {
        "feature": feature,
        "peak_position": int(peak_row["position"]),
        "peak_value": float(peak_row["difference"]),
        "peak_type": "increase" if peak_row["difference"] > 0 else "decrease",
    }


# Detect peaks for each feature
print("=" * 80)
print("PEAK DETECTION - SIGNATURE LOCALIZATION")
print("=" * 80)

peaks = []
for feature in features_to_plot:
    peak_info = find_signature_peak(diff_df, feature, window=(-25, 25))
    peaks.append(peak_info)
    
    print(f"\n{feature.upper()}:")
    print(f"  Peak at position: {peak_info['peak_position']:+d} nt")
    print(f"  Peak value:       {peak_info['peak_value']:+.4f}")
    print(f"  Direction:        {peak_info['peak_type']}")

peaks_df = pl.DataFrame(peaks)

# Calculate offset
dwell_peak = peaks_df.filter(pl.col("feature") == "dwell_time")["peak_position"][0]
signal_peak = peaks_df.filter(pl.col("feature") == "signal_mean")["peak_position"][0]

measured_offset = abs(dwell_peak - signal_peak)

print("\n" + "=" * 80)
print("MEASURED OFFSET")
print("=" * 80)
print(f"Dwell peak position:   {dwell_peak:+d} nt")
print(f"Current peak position: {signal_peak:+d} nt")
print(f"\n➤ OFFSET MAGNITUDE:    {measured_offset} nt")
print(f"  Expected offset:     ~12-13 nt")

if measured_offset >= 10 and measured_offset <= 15:
    print(f"\n✓ Measured offset ({measured_offset} nt) is consistent with expected ~12-13 nt!")
else:
    print(f"\n⚠️  Measured offset ({measured_offset} nt) differs from expected ~12-13 nt")

# Determine offset direction
if dwell_peak > signal_peak:
    direction = "downstream (toward 3' end / adaptor)"
    offset_sign = "+"
elif dwell_peak < signal_peak:
    direction = "upstream (toward 5' end / tRNA body)"
    offset_sign = "-"
else:
    direction = "aligned (no offset detected)"
    offset_sign = "0"

print(f"\n➤ OFFSET DIRECTION: Dwell signature is {direction}")
print(f"  Dwell at position:   {dwell_peak:+d}")
print(f"  Current at position: {signal_peak:+d}")

# Save offset measurements
offset_report = {
    "measured_offset_nt": measured_offset,
    "dwell_peak_position": dwell_peak,
    "current_peak_position": signal_peak,
    "offset_direction": direction,
    "offset_sign": offset_sign,
    "expected_offset_nt": "12-13",
}

import json
with open(OUTPUT_DIR / "offset_measurements.json", "w") as f:
    json.dump(offset_report, f, indent=2)

print(f"\n✓ Saved offset measurements: {OUTPUT_DIR / 'offset_measurements.json'}")

## Step 6: Heatmap Visualization

Create a 2D heatmap showing all features across position for charged vs uncharged.

In [ ]:
# Create bar plots for charged and uncharged side-by-side
colors = {"charged": "#E74C3C", "uncharged": "#3498DB"}

for feature in features_to_plot:
    feature_data = stats_pd[stats_pd["feature"] == feature].copy()
    
    plot = (
        p9.ggplot(feature_data, p9.aes(x="position", y="mean", fill="label"))
        + p9.geom_col(alpha=0.7, width=0.8, position="identity")
        + p9.geom_vline(xintercept=0, linetype="dashed", color="black", alpha=0.5, size=1.5)
        + p9.scale_fill_manual(values=colors)
        + p9.facet_wrap("~label", ncol=1)
        + p9.labs(
            title=f"{titles[feature]} - By Sample",
            x="Position Relative to Motif (nt)",
            y=feature.replace("_", " ").title(),
            fill="Sample"
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(14, 8),
            plot_title=p9.element_text(size=12, weight="bold"),
            strip_text=p9.element_text(size=11, weight="bold"),
            legend_position="none"
        )
    )
    
    # Save plot
    plot.save(OUTPUT_DIR / f"{feature}_by_sample.png", dpi=150, width=14, height=8)
    print(plot)

print(f"\n✓ Saved faceted plots: {OUTPUT_DIR}/{{feature}}_by_sample.png")

## Step 7: Cross-Correlation Analysis

Compute cross-correlation between dwell and current difference signals to quantify the offset.

In [ ]:
# Extract difference signals
dwell_diff = diff_df[diff_df["feature"] == "dwell_time"].sort_values("position")
signal_diff = diff_df[diff_df["feature"] == "signal_mean"].sort_values("position")

# Compute cross-correlation
correlation = sp_signal.correlate(dwell_diff["difference"], signal_diff["difference"], mode="full")
lags = sp_signal.correlation_lags(
    len(dwell_diff["difference"]),
    len(signal_diff["difference"]),
    mode="full"
)

# Find peak correlation
peak_idx = np.argmax(np.abs(correlation))
peak_lag = lags[peak_idx]
peak_corr = correlation[peak_idx]

print("=" * 80)
print("CROSS-CORRELATION ANALYSIS")
print("=" * 80)
print(f"Peak correlation: {peak_corr:.4f}")
print(f"Peak lag:         {peak_lag} positions")
print(f"\nInterpretation: Dwell signal is shifted by {peak_lag} positions relative to current signal")

# Create dataframe for plotting
corr_df = pl.DataFrame({
    "lag": lags,
    "correlation": correlation
}).to_pandas()

# Plot cross-correlation
plot = (
    p9.ggplot(corr_df, p9.aes(x="lag", y="correlation"))
    + p9.geom_line(color="#9B59B6", size=1.2, alpha=0.8)
    + p9.geom_vline(xintercept=peak_lag, linetype="dashed", color="red", size=1.2)
    + p9.geom_vline(xintercept=0, linetype="solid", color="black", alpha=0.3, size=0.8)
    + p9.geom_hline(yintercept=0, linetype="solid", color="black", alpha=0.3, size=0.8)
    + p9.annotate(
        "text",
        x=peak_lag,
        y=peak_corr,
        label=f"Peak at lag={peak_lag}",
        ha="left",
        va="bottom",
        color="red",
        size=10
    )
    + p9.labs(
        title="Cross-Correlation: Dwell vs Current Difference Signals",
        x="Lag (positions)",
        y="Cross-Correlation"
    )
    + p9.theme_minimal()
    + p9.theme(
        figure_size=(14, 6),
        plot_title=p9.element_text(size=13, weight="bold")
    )
)

# Save plot
plot.save(OUTPUT_DIR / "cross_correlation.png", dpi=150, width=14, height=6)
print(plot)

print(f"\n✓ Saved cross-correlation plot: {OUTPUT_DIR / 'cross_correlation.png'}")

## Summary and Recommendations

Based on the analysis above, we can now make data-driven decisions about implementing offset correction.

In [ ]:
print("=" * 80)
print("SUMMARY & RECOMMENDATIONS")
print("=" * 80)

print("\n1. MEASURED OFFSET:")
print(f"   Magnitude: {measured_offset} nt")
print(f"   Direction: {direction}")
print(f"   Dwell peak: {dwell_peak:+d} nt from motif")
print(f"   Current peak: {signal_peak:+d} nt from motif")

print("\n2. BIOLOGICAL INTERPRETATION:")
if dwell_peak > 0 and signal_peak == 0:
    print("   ✓ Dwell signature appears downstream (toward adaptor/3' end)")
    print("   ✓ Current signature is at the motif (amino acid position)")
    print("   ✓ This suggests: When aa is in MOTOR (at motif+13), nucleotides")
    print("     at motif position are in PORE (experiencing dwell increase)")
elif dwell_peak < 0 and signal_peak == 0:
    print("   ✓ Dwell signature appears upstream (toward tRNA body/5' end)")
    print("   ✓ Current signature is at the motif (amino acid position)")
elif dwell_peak == 0 and signal_peak == 0:
    print("   ⚠️  Both signatures at motif - unexpected!")
    print("   This contradicts the motor-pore offset hypothesis")
else:
    print(f"   Dwell at {dwell_peak:+d}, Current at {signal_peak:+d}")

print("\n3. IMPLEMENTATION RECOMMENDATIONS:")
print("\n   A. IMMEDIATE ACTION:")
print(f"      Modify chunking/extractor.py to extract features from offset windows:")
print(f"      - Dwell features: base_idx + {dwell_peak} (±5 nt context)")
print(f"      - Current features: base_idx + {signal_peak} (±5 nt context)")

print("\n   B. CONFIGURATION:")
print("      Add parameters to constants.py:")
print(f"      - DWELL_OFFSET = {dwell_peak}")
print(f"      - CURRENT_OFFSET = {signal_peak}")

print("\n   C. MODEL ARCHITECTURE:")
if measured_offset > 10:
    print("      Option 1 (Recommended): Separate feature windows")
    print("        - Extract dwell and current from different positions")
    print("        - Keep current model architecture (feature concatenation)")
    print("")
    print("      Option 2: Wider context window")
    print(f"        - Use ±{measured_offset + 5} nt context to capture both signatures")
    print("        - Let model learn to use both via convolutions/attention")
else:
    print("      Current 11-base window (±5 nt) may be sufficient")
    print("      Small offset can be handled by convolutional receptive field")

print("\n   D. VALIDATION:")
print("      1. Retrain models with offset-corrected features")
print("      2. Compare accuracy: baseline vs offset-aware")
print("      3. EXPECT: Dwell features now contribute positively to accuracy")
print("      4. Analyze learned representations to confirm offset capture")

print("\n4. NEXT STEPS:")
print("   [ ] Implement offset parameters in chunking/extractor.py")
print("   [ ] Update CLI to support --dwell-offset and --current-offset")
print("   [ ] Prepare new training data with corrected offsets")
print("   [ ] Retrain ConvLSTMDwell model")
print("   [ ] Compare old vs new model performance")
print("   [ ] Document findings in GitHub issue #81")

print("\n" + "=" * 80)
print("OUTPUT FILES:")
print("=" * 80)
print(f"  {OUTPUT_DIR / 'position_wise_profiles.png'}")
print(f"  {OUTPUT_DIR / 'difference_profiles.png'}")
print(f"  {OUTPUT_DIR / 'feature_heatmaps.png'}")
print(f"  {OUTPUT_DIR / 'cross_correlation.png'}")
print(f"  {OUTPUT_DIR / 'offset_measurements.json'}")
print("=" * 80)